# COMP3010 Assignment — BLEVE Blast Pressure Prediction

**Task.** Regression. Predict `Target Pressure (bar)` — the peak blast overpressure
recorded at a virtual sensor — from 23 scenario features describing a *Boiling Liquid
Expanding Vapour Explosion* (BLEVE): tank geometry, substance thermodynamics, obstacle
layout, and sensor position.

**Evaluation.** Mean Absolute Percentage Error (MAPE, primary) and the coefficient of
determination (R², required). RMSE is reported for context.

**Models compared.**
1. **Ridge Regression** — linear baseline with L2 regularisation (grid-searched α).
2. **XGBoost** — gradient-boosted trees with randomised hyper-parameter search.
3. **MLP (PyTorch)** — 3×256 feed-forward net with Mish activations and a Softplus head,
   following the architecture used by Li et al. (2021) for BLEVE overpressure regression.
4. **FT-Transformer (PyTorch, bonus)** — Feature Tokenizer + Transformer as per Li et al.
   (2023). Implemented from scratch using the exact architecture and hyper-parameters
   from their Table 5 (4 layers, 72-dim tokens, 8 heads, ReGLU FFN).

**Key methodological choice.** The target is *heavily right-skewed* (min ≈ 0.016 bar,
max ≈ 9.17 bar, median ≈ 0.21 bar). We apply a **Gaussian quantile transform** to the
target before fitting, and invert it for prediction. This is the same technique used by
Li et al. (2023) in their FT-Transformer BLEVE paper, and it markedly improves MAPE
on rare high-pressure events by giving the learner a roughly-normal response surface.

**Repro.** Every stochastic step uses `random_state=42`. The notebook runs end-to-end
on Google Colab — see the first cell for dependency installation.

## 1. Setup & imports

Install any non-standard packages (`xgboost` is pre-installed on Colab but we pin it defensively). Then import everything we'll need and fix seeds.

In [ ]:
# Uncomment on Colab if these are missing. On a fresh Colab runtime
# xgboost and torch are already available; pandas/sklearn/seaborn too.
# !pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn torch

Import every library the notebook needs, fix the global seed to 42, and select GPU if available.

In [ ]:
import os, sys, math, random, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_squared_error

import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python     :", sys.version.split()[0])
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("XGBoost    :", xgb.__version__)
print("PyTorch    :", torch.__version__, "| device:", DEVICE)

sns.set_theme(style="whitegrid", context="notebook")

## 2. Data loading & exploration

Locate `train.csv`, `test.csv`, and `sample_prediction.csv`. The cell below tries three
common locations so the notebook works both on Colab (mounted Drive) and locally.

In [ ]:
# Try common data locations in order. Adjust DATA_DIR manually if none match.
_candidates = [
    Path("/content/drive/MyDrive/COMP3010"),
    Path("/content"),
    Path("/mnt/project"),
    Path("."),
]
DATA_DIR = next((p for p in _candidates if (p / "train.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find train.csv — set DATA_DIR manually.")
print("Using DATA_DIR =", DATA_DIR)

train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw  = pd.read_csv(DATA_DIR / "test.csv")
sample    = pd.read_csv(DATA_DIR / "sample_prediction.csv")

print("train :", train_raw.shape)
print("test  :", test_raw.shape)
print("sample:", sample.shape)

Quick look at the schema and a few rows.

In [ ]:
train_raw.head()

Column dtypes and non-null counts.

In [ ]:
train_raw.info()

Summary statistics. Watch for suspicious minima (e.g. *negative* BLEVE height) and the -1/-42 sentinels in `Liquid Boiling Temperature (K)`.

In [ ]:
train_raw.describe().T

Categorical inspection. `Status` should be a 2-level variable (Superheated / Subcooled) but the raw data contains several typo variants that must be mapped before one-hot encoding.

In [ ]:
print("Status raw value counts:")
print(train_raw["Status"].value_counts(dropna=False))

Missing-value heatmap. NaNs are sparse (~0.1% per column) but present in every feature — imputation is required.

In [ ]:
plt.figure(figsize=(12, 4))
sns.heatmap(train_raw.isna(), cbar=False, yticklabels=False)
plt.title("Training-set missingness (white = NaN)")
plt.tight_layout(); plt.show()

print("Total NaNs in train :", int(train_raw.isna().sum().sum()))
print("Total NaNs in test  :", int(test_raw.isna().sum().sum()))

Target distribution. Notice the long right tail — this motivates the quantile transform later.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train_raw["Target Pressure (bar)"].dropna(), bins=80, ax=axes[0])
axes[0].set_title("Target pressure (raw)")
axes[0].set_xlabel("Pressure (bar)")
sns.histplot(np.log1p(train_raw["Target Pressure (bar)"].dropna()), bins=80, ax=axes[1], color="darkorange")
axes[1].set_title("Target pressure (log1p)")
axes[1].set_xlabel("log1p(Pressure)")
plt.tight_layout(); plt.show()

## 3. Data cleaning

Cleaning decisions, in order, and why:

1. **Drop the 10 rows with NaN target** — we cannot learn from them and they must not leak into validation MAPE (dividing by a tiny imputed target would explode the metric).
2. **Drop full-row duplicates** (≈ 50 rows).
3. **Normalise `Status`** — map typo variants (`Subcool`, `subcooled`, `Saperheated`, …) onto the two canonical classes, then encode as a binary indicator.
4. **Replace sentinel values in `Liquid Boiling Temperature (K)`** — values of `-1` and `-42` are missing-data flags (the physical boiling point of every fluid here is well above 0 K). We mask them as NaN and impute.
5. **Clip non-physical `BLEVE Height`** — 191 training rows have tiny negative heights (≈ -0.02 m) which is physically impossible (a BLEVE happens above ground level). We clip at 0.
6. **Generic outlier guard** — for every numeric feature, clip values that fall outside `[Q1 - 3·IQR, Q3 + 3·IQR]`. This is a defensive rule that catches any further garbage (e.g. the extreme magnitudes sometimes seen in this dataset) without removing legitimate variation.
7. **Impute remaining NaNs** — median for numerics, mode for categoricals — fitted **on training data only** and reused for the test set.

In [ ]:
# --- 3a. Drop rows with missing target --------------------------------
df = train_raw.copy()
n0 = len(df)
df = df.dropna(subset=["Target Pressure (bar)"]).reset_index(drop=True)
print(f"Dropped {n0 - len(df)} rows with NaN target.  Remaining: {len(df)}")

# --- 3b. Drop exact duplicates ---------------------------------------
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {n0 - len(df)} full-row duplicates. Remaining: {len(df)}")

**3c.** Canonicalise `Status` — map typo variants onto the two true classes.

In [ ]:
# --- 3c. Normalise Status -------------------------------------------
# Map typo variants. Anything starting with a "super"-like prefix → Superheated,
# everything else → Subcooled. Case-insensitive.
def canonicalise_status(s):
    if pd.isna(s):
        return np.nan
    s_low = str(s).strip().lower()
    # handful of obvious "superheated" misspellings
    super_markers = ("super", "saper")
    return "Superheated" if s_low.startswith(super_markers) else "Subcooled"

df["Status"] = df["Status"].map(canonicalise_status)
test_clean   = test_raw.copy()
test_clean["Status"] = test_clean["Status"].map(canonicalise_status)
print("Status after normalisation (train):")
print(df["Status"].value_counts(dropna=False))
print("\nStatus after normalisation (test):")
print(test_clean["Status"].value_counts(dropna=False))

**3d.** Mask the `-1` and `-42` sentinel values in `Liquid Boiling Temperature (K)` so they are treated as NaN by the imputer.

In [ ]:
# --- 3d. Boiling-temperature sentinels ------------------------------
for frame in (df, test_clean):
    col = "Liquid Boiling Temperature (K)"
    frame.loc[frame[col] <= 0, col] = np.nan
print("train NaNs in boiling temp after masking:", df["Liquid Boiling Temperature (K)"].isna().sum())
print("test  NaNs in boiling temp after masking:", test_clean["Liquid Boiling Temperature (K)"].isna().sum())

**3e.** Clip the ~200 rows with physically-impossible negative BLEVE Height to 0.

In [ ]:
# --- 3e. Clip non-physical BLEVE Height -----------------------------
neg_mask = df["BLEVE Height (m)"] < 0
print(f"Clipping {neg_mask.sum()} rows with negative BLEVE Height to 0.0")
df.loc[neg_mask, "BLEVE Height (m)"] = 0.0
test_clean.loc[test_clean["BLEVE Height (m)"] < 0, "BLEVE Height (m)"] = 0.0

**3f.** Generic IQR-based outlier clipping — a safety net for any remaining garbage values.

In [ ]:
# --- 3f. Generic IQR-based outlier clipping -------------------------
# We CLIP (not drop) so row count stays the same. Clipping bounds come from
# the training distribution only — applied identically to test.
NUMERIC_COLS = [c for c in df.columns
                if c not in ("ID", "Status", "Target Pressure (bar)")
                and pd.api.types.is_numeric_dtype(df[c])]

clip_bounds = {}
for c in NUMERIC_COLS:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    clip_bounds[c] = (lo, hi)
    before_hi = (df[c] > hi).sum()
    before_lo = (df[c] < lo).sum()
    df[c] = df[c].clip(lo, hi)
    test_clean[c] = test_clean[c].clip(lo, hi)

n_clipped = sum((train_raw[c] > clip_bounds[c][1]).sum() + (train_raw[c] < clip_bounds[c][0]).sum()
                for c in NUMERIC_COLS if c in train_raw.columns)
print(f"Total cell-level clips applied across training set: ~{n_clipped}")

**3g.** Impute remaining NaNs — median for numeric features, mode for `Status`. Imputers are fit on train only and reused for test.

In [ ]:
# --- 3g. Impute remaining NaNs --------------------------------------
# Fit imputers on train only, then transform both train and test.
#
# keep_empty_features=True ensures that a column that happens to be entirely
# NaN in the fit data is *kept* (imputed with 0 / mode) rather than silently
# dropped — otherwise the column count of the output array no longer matches
# NUMERIC_COLS and the assignment below fails.
num_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
cat_imputer = SimpleImputer(strategy="most_frequent", keep_empty_features=True)

df[NUMERIC_COLS] = pd.DataFrame(
    num_imputer.fit_transform(df[NUMERIC_COLS]),
    columns=NUMERIC_COLS, index=df.index,
)
test_clean[NUMERIC_COLS] = pd.DataFrame(
    num_imputer.transform(test_clean[NUMERIC_COLS]),
    columns=NUMERIC_COLS, index=test_clean.index,
)

df[["Status"]] = pd.DataFrame(
    cat_imputer.fit_transform(df[["Status"]]),
    columns=["Status"], index=df.index,
)
test_clean[["Status"]] = pd.DataFrame(
    cat_imputer.transform(test_clean[["Status"]]),
    columns=["Status"], index=test_clean.index,
)

print("Train NaNs remaining:", int(df.isna().sum().sum()))
print("Test  NaNs remaining:", int(test_clean.isna().sum().sum()))

## 4. Feature engineering

Four physically-motivated derived features:

- **`sensor_distance`** — 3-D Euclidean distance from the tank to the sensor. Blast overpressure falls roughly as `1/r` for the first few metres, so distance is the single most predictive engineered feature.
- **`tank_volume`** — product of tank W×L×H, proxy for the total stored energy driving the blast.
- **`tank_aspect`** — Length/Width, distinguishes long cylindrical tanks from cubic ones (affects directionality of the blast wave).
- **`vapour_fraction`** — Vapour Height / Tank Height, the fraction of the tank that is already gas; a higher fraction typically means a weaker liquid-phase BLEVE.

We also **one-hot encode** `Status` as a single binary column (`is_superheated`).

In [ ]:
def engineer(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()

    # 3-D sensor distance (sqrt(x² + y² + z²))
    f["sensor_distance"] = np.sqrt(
        f["Sensor Position x"] ** 2
        + f["Sensor Position y"] ** 2
        + f["Sensor Position z"] ** 2
    )

    # tank volume
    f["tank_volume"] = f["Tank Width (m)"] * f["Tank Length (m)"] * f["Tank Height (m)"]

    # aspect ratio (guard against zero division — test data is clean, train is clipped)
    f["tank_aspect"] = f["Tank Length (m)"] / f["Tank Width (m)"].replace(0, np.nan)
    f["tank_aspect"] = f["tank_aspect"].fillna(f["tank_aspect"].median())

    # vapour fraction
    f["vapour_fraction"] = f["Vapour Height (m)"] / f["Tank Height (m)"].replace(0, np.nan)
    f["vapour_fraction"] = f["vapour_fraction"].fillna(f["vapour_fraction"].median())

    # Status → binary
    f["is_superheated"] = (f["Status"] == "Superheated").astype(int)
    f = f.drop(columns=["Status"])
    return f

df_fe         = engineer(df)
test_clean_fe = engineer(test_clean)

print("Feature-engineered training shape:", df_fe.shape)
print("New columns:", ["sensor_distance", "tank_volume", "tank_aspect",
                       "vapour_fraction", "is_superheated"])
df_fe.head(3)

## 5. Preprocessing pipeline

- Split features `X` from target `y`, drop the `ID` column (it is an identifier only, not a feature).
- Hold out 20 % of training for validation (stratified by a 5-bin split of the target to keep all pressure regimes represented).
- **StandardScaler** on `X` — fit on train only, transform val and test.
- **QuantileTransformer** (output_distribution="normal") on `y`. This is the single most important preprocessing step for this regression: the raw target spans nearly three orders of magnitude and is highly right-skewed, so fitting regression models directly on it gives huge errors on the long tail. Mapping `y` to a Gaussian gives every model a smooth, well-conditioned target.

In [ ]:
# Features / target
drop_cols = ["ID", "Target Pressure (bar)"]
FEATURES = [c for c in df_fe.columns if c not in drop_cols]
print(f"Using {len(FEATURES)} features:")
for i, c in enumerate(FEATURES, 1):
    print(f"  {i:2d}. {c}")

X = df_fe[FEATURES].values.astype(np.float32)
y = df_fe["Target Pressure (bar)"].values.astype(np.float32)
X_test = test_clean_fe[FEATURES].values.astype(np.float32)
test_ids = test_clean_fe["ID"].values

Stratified 80/20 train/validation split. We bin the continuous target into 5 quantile groups so both splits see the full pressure range.

In [ ]:
# Stratified 80/20 train/val split using 5 equal-frequency target bins
# (regression-friendly stratification)
y_bins = pd.qcut(y, q=5, labels=False, duplicates="drop")

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y_bins,
)
print(f"Train: {X_tr.shape}  |  Val: {X_val.shape}")

Scale features (StandardScaler) and map the target into an approximately-Gaussian space via QuantileTransformer. Plot the before/after target distribution.

In [ ]:
# Scale features (fit on train only)
scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# Quantile-transform target into Gaussian space.
qt = QuantileTransformer(
    output_distribution="normal",
    n_quantiles=min(1000, len(y_tr)),
    random_state=SEED,
)
y_tr_q  = qt.fit_transform(y_tr.reshape(-1, 1)).ravel()
y_val_q = qt.transform(y_val.reshape(-1, 1)).ravel()

# Visualise the transform
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y_tr, bins=80, ax=axes[0])
axes[0].set_title("Target — original scale (train)")
axes[0].set_xlabel("Pressure (bar)")
sns.histplot(y_tr_q, bins=80, ax=axes[1], color="seagreen")
axes[1].set_title("Target — after quantile transform")
axes[1].set_xlabel("z (approx. N(0,1))")
plt.tight_layout(); plt.show()

Helper for evaluation. All three metrics are computed in the *original* pressure units — MAPE only makes sense on the physical scale, and the whole point of the target transform is that we invert it at prediction time.

In [ ]:
def evaluate(y_true_orig, y_pred_orig, label=""):
    mape = mean_absolute_percentage_error(y_true_orig, y_pred_orig)
    r2   = r2_score(y_true_orig, y_pred_orig)
    rmse = math.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    if label:
        print(f"{label:<35} MAPE={mape:7.4f}  R²={r2:7.4f}  RMSE={rmse:7.4f}")
    return {"MAPE": mape, "R2": r2, "RMSE": rmse}

def inverse_target(y_q):
    """Map a quantile-space prediction back to bar."""
    return qt.inverse_transform(np.asarray(y_q).reshape(-1, 1)).ravel()

# Container for every model's results
results = {}

## 6. Model 1 — Ridge Regression

Linear baseline. Grid-search α over `[0.01, 0.1, 1, 10, 100]` with 5-fold CV on the
quantile-transformed target.

In [ ]:
ridge_grid = GridSearchCV(
    estimator=Ridge(random_state=SEED),
    param_grid={"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
ridge_grid.fit(X_tr_s, y_tr_q)
print("Best α :", ridge_grid.best_params_["alpha"])
ridge = ridge_grid.best_estimator_

Score Ridge on train & validation (in original pressure units), and cache its test-set predictions in quantile space for later export.

In [ ]:
# Evaluate on train / val / test-of-train (here the held-out 20 %)
pred_tr_orig  = inverse_target(ridge.predict(X_tr_s))
pred_val_orig = inverse_target(ridge.predict(X_val_s))

results["Ridge"] = {
    "train": evaluate(y_tr,  pred_tr_orig,  "Ridge (train)"),
    "val":   evaluate(y_val, pred_val_orig, "Ridge (val)  "),
}

# Keep quantile-space predictions for test too (used later if Ridge wins)
ridge_test_pred_q = ridge.predict(X_test_s)

## 7. Model 2 — XGBoost

Randomised search over the five most impactful gradient-boosting hyper-parameters, with
5-fold CV on the quantile-transformed target. We use a modest number of iterations
(`n_iter=25`) to keep runtime reasonable in Colab while still exploring the space.

In [ ]:
xgb_param_dist = {
    "n_estimators":     [100, 200, 400, 600, 800, 1000],
    "max_depth":        [3, 4, 5, 6, 7, 8, 9, 10],
    "learning_rate":    [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3],
    "subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
}

xgb_base = xgb.XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=25,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    random_state=SEED,
    verbose=0,
)

t0 = time.time()
# Note: XGBoost is fine with unscaled features, but feeding the scaled ones keeps the
# pipeline identical to the NN's. Trees are scale-invariant so this has no effect.
xgb_search.fit(X_tr_s, y_tr_q)
print(f"XGBoost RandomizedSearchCV finished in {time.time() - t0:.1f} s")
print("Best params:", xgb_search.best_params_)

xgbm = xgb_search.best_estimator_

Score XGBoost on train & validation in original pressure units, and cache its test-set predictions in quantile space.

In [ ]:
pred_tr_orig  = inverse_target(xgbm.predict(X_tr_s))
pred_val_orig = inverse_target(xgbm.predict(X_val_s))

results["XGBoost"] = {
    "train": evaluate(y_tr,  pred_tr_orig,  "XGBoost (train)"),
    "val":   evaluate(y_val, pred_val_orig, "XGBoost (val)  "),
}

xgb_test_pred_q = xgbm.predict(X_test_s)

Which features drive the XGBoost predictions?

In [ ]:
imp = pd.Series(xgbm.feature_importances_, index=FEATURES).sort_values(ascending=True)
plt.figure(figsize=(8, 7))
imp.plot(kind="barh")
plt.title("XGBoost — feature importance (gain)")
plt.xlabel("Importance")
plt.tight_layout(); plt.show()

## 8. Model 3 — MLP Neural Network (PyTorch)

Architecture — **3 hidden layers × 256 neurons**, matching Li et al. (2021)'s BLEVE
over-pressure regression model:

- **Mish** activation on every hidden layer (`x * tanh(softplus(x))`). Smoother than
  ReLU, empirically helps tabular regression.
- **Softplus** activation on the output. Guarantees a strictly positive prediction,
  which matches the physical constraint *P ≥ 0*.
- **Dropout 0.1** between hidden layers.
- **L2 weight decay** via the SGD optimiser.

Training —

- **SGD, momentum 0.9**, batch size 512, up to 500 epochs.
- **ReduceLROnPlateau** (patience 50 epochs, factor 0.5).
- **Early stopping** after 80 epochs without validation improvement.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=256, n_layers=3, dropout=0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(prev, hidden), nn.Mish(), nn.Dropout(dropout)]
            prev = hidden
        layers += [nn.Linear(prev, 1), nn.Softplus()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

The Softplus output layer means the network produces *strictly positive* values. But
our target is quantile-transformed into an approximately N(0,1) space that includes
negative numbers. To reconcile the two we **shift the quantile target by a fixed offset**
(the training-set minimum, minus a small margin) before fitting, and subtract the offset
back at prediction time. This keeps the Softplus constraint while letting the loss see
negative "logical" targets.

In [ ]:
# Shift constant so that the shifted quantile target is strictly > 0.
Y_SHIFT = float(-y_tr_q.min() + 1.0)   # e.g. ~+4.5
print(f"Target shift constant = {Y_SHIFT:.4f}")

y_tr_q_shift  = y_tr_q  + Y_SHIFT
y_val_q_shift = y_val_q + Y_SHIFT

# Build tensors
t_Xtr  = torch.from_numpy(X_tr_s).float()
t_ytr  = torch.from_numpy(y_tr_q_shift).float()
t_Xval = torch.from_numpy(X_val_s).float()
t_yval = torch.from_numpy(y_val_q_shift).float()

BATCH = 512
train_loader = DataLoader(TensorDataset(t_Xtr, t_ytr), batch_size=BATCH,
                          shuffle=True, drop_last=False)
val_tensor_X = t_Xval.to(DEVICE); val_tensor_y = t_yval.to(DEVICE)

Instantiate the MLP and run the training loop with SGD + momentum, LR scheduling, and early stopping. Progress is printed every 25 epochs.

In [ ]:
model = MLP(in_dim=X_tr_s.shape[1], hidden=256, n_layers=3, dropout=0.1).to(DEVICE)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4,      # L2 regularisation
    nesterov=True,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=50, min_lr=1e-5,
)
loss_fn = nn.MSELoss()

MAX_EPOCHS      = 500
EARLY_PATIENCE  = 80

best_val, best_state, epochs_no_imp = float("inf"), None, 0
history = {"train_loss": [], "val_loss": [], "lr": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb); n += len(xb)
    train_loss = epoch_loss / n

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(val_tensor_X), val_tensor_y).item()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(optimizer.param_groups[0]["lr"])
    scheduler.step(val_loss)

    if val_loss < best_val - 1e-5:
        best_val, best_state, epochs_no_imp = val_loss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
    else:
        epochs_no_imp += 1

    if epoch == 1 or epoch % 25 == 0 or epoch == MAX_EPOCHS:
        print(f"epoch {epoch:3d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}"
              f"  lr={optimizer.param_groups[0]['lr']:.2e}")

    if epochs_no_imp >= EARLY_PATIENCE:
        print(f"Early stopping at epoch {epoch} (no improvement for {EARLY_PATIENCE} epochs)")
        break

model.load_state_dict(best_state)
print(f"Best val loss (shifted-quantile MSE) = {best_val:.5f}")

Plot training / validation loss curves alongside the learning-rate schedule to confirm the model converged and the LR reduced as expected.

In [ ]:
# Training curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"],   label="val")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("MSE (quantile-shift space)")
ax[0].set_title("MLP loss"); ax[0].legend()
ax[1].plot(history["lr"])
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("learning rate")
ax[1].set_title("LR schedule"); ax[1].set_yscale("log")
plt.tight_layout(); plt.show()

Score the MLP on train & validation in original pressure units, and cache its test-set predictions in quantile space.

In [ ]:
@torch.no_grad()
def mlp_predict_orig(X_scaled):
    model.eval()
    t = torch.from_numpy(X_scaled).float().to(DEVICE)
    # batch to avoid GPU OOM on big sets
    out = []
    for i in range(0, len(t), 4096):
        out.append(model(t[i:i+4096]).cpu().numpy())
    y_shift = np.concatenate(out)
    y_q     = y_shift - Y_SHIFT                # undo the shift
    return inverse_target(y_q)                 # undo the quantile transform

pred_tr_orig  = mlp_predict_orig(X_tr_s)
pred_val_orig = mlp_predict_orig(X_val_s)

results["MLP"] = {
    "train": evaluate(y_tr,  pred_tr_orig,  "MLP (train)"),
    "val":   evaluate(y_val, pred_val_orig, "MLP (val)  "),
}

# Quantile-space test predictions for final export, after shift removal
_t_test = torch.from_numpy(X_test_s).float().to(DEVICE)
with torch.no_grad():
    mlp_test_pred_q = (model(_t_test).cpu().numpy()) - Y_SHIFT

## 8b. Model 4 (bonus) — FT-Transformer

The *Feature Tokenizer + Transformer*, as used by Li et al. (2023) — the paper from
which this assignment's BLEVE dataset and methodology are drawn. The authors found FT-T
to be the best-performing model among LightGBM / MLP / ResNet / FT-Transformer on the
same dataset, reaching a relative error of ~3.5 % and R² ≈ 0.997.

**Architecture** (Li et al. eq. 3, Fig. 4-5):

- **Feature Tokenizer.** Each of the `n` numeric features is independently projected
  into a `d`-dimensional embedding by its own trainable `(weight, bias)` pair. A
  learnable `[CLS]` token is appended, giving `(n + 1)` tokens per sample.
- **Transformer blocks.** Each block is two residual sub-blocks using *pre-norm*:
  `y = x + Dropout(Module(LayerNorm(x)))` where *Module* is first Multi-Head
  Self-Attention, then a feed-forward network.
- **Prediction head.** `Linear(ReLU(LayerNorm(x_CLS)))` — only the `[CLS]` token is
  used for the final pressure prediction, with Softplus to enforce positivity.

**Hyper-parameters** (Li et al. Table 5 — their optimal config, lightly adapted to
our 27-feature input):

| Param | Value |
|---|---|
| n_layers | 4 |
| token_dim *d* | 72 |
| attention heads | 8 |
| dropout | 0.04 |
| FFN hidden factor | 2 × d |
| learning rate | 1e-4 |
| weight decay | 2.38e-6 |
| batch size | 512 |
| optimiser | AdamW |
| loss | MSE (on quantile-shifted target) |

Note on the activation: Li et al. report "ReGLU" (Gated Linear Unit with ReLU gate) in
the FFN. We implement this faithfully.

Define the FT-Transformer modules: a per-feature tokenizer, a ReGLU FFN, a pre-norm transformer block, and the full model with Softplus head.

In [ ]:
class FeatureTokenizer(nn.Module):
    """Per-feature linear projection to a d-dim token, plus learnable [CLS] token.

    For n features, we keep n separate (weight, bias) pairs:
        token_i = x_i * W_i + b_i     (W_i, b_i ∈ R^d)
    This is exactly the formulation in Gorishniy et al. / Li et al. eq. 3.
    """
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias   = nn.Parameter(torch.empty(n_features, d_token))
        self.cls    = nn.Parameter(torch.empty(1, 1, d_token))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.bias,   a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.cls,    a=math.sqrt(5))

    def forward(self, x):               # x: (B, n)
        x = x.unsqueeze(-1) * self.weight + self.bias   # (B, n, d)
        cls = self.cls.expand(x.size(0), -1, -1)        # (B, 1, d)
        return torch.cat([cls, x], dim=1)               # (B, n+1, d)


class ReGLU(nn.Module):
    """ReGLU activation: split the last dim in half, (x1, x2) → x1 * ReLU(x2).

    Following Shazeer (2020) "GLU Variants Improve Transformer" and used by
    Li et al. 2023 in their BLEVE FT-Transformer.
    """
    def forward(self, x):
        a, b = x.chunk(2, dim=-1)
        return a * torch.relu(b)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block: y = x + Dropout(Module(Norm(x))).

    MHSA sub-block followed by an FFN sub-block, both with residual connections.
    """
    def __init__(self, d_token: int, n_heads: int, ffn_hidden: int, dropout: float):
        super().__init__()
        self.norm_attn = nn.LayerNorm(d_token)
        self.attn      = nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
        self.drop_attn = nn.Dropout(dropout)

        self.norm_ffn  = nn.LayerNorm(d_token)
        # ReGLU halves the channel count, so inner linear outputs 2 * ffn_hidden
        self.ffn       = nn.Sequential(
            nn.Linear(d_token, 2 * ffn_hidden),
            ReGLU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_hidden, d_token),
        )
        self.drop_ffn  = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm_attn(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop_attn(h)

        h = self.norm_ffn(x)
        h = self.ffn(h)
        x = x + self.drop_ffn(h)
        return x


class FTTransformer(nn.Module):
    """Feature Tokenizer + Transformer for tabular regression.

    Final prediction is taken from the [CLS] token only (index 0 after tokenisation).
    Softplus head enforces a strictly-positive pressure prediction, matching the
    physical constraint of the target.
    """
    def __init__(self, n_features, d_token=72, n_layers=4, n_heads=8,
                 ffn_factor=2.0, dropout=0.04):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        ffn_hidden = int(d_token * ffn_factor)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_token, n_heads, ffn_hidden, dropout)
            for _ in range(n_layers)
        ])
        self.head_norm = nn.LayerNorm(d_token)
        self.head = nn.Sequential(
            nn.ReLU(),
            nn.Linear(d_token, 1),
            nn.Softplus(),
        )

    def forward(self, x):
        tok = self.tokenizer(x)                # (B, n+1, d)
        for blk in self.blocks:
            tok = blk(tok)
        cls = tok[:, 0, :]                     # extract [CLS] token
        return self.head(self.head_norm(cls)).squeeze(-1)

Train the FT-Transformer with AdamW on MSE loss over the quantile-shifted target. Use the same early-stopping + LR scheduling protocol as the MLP.

In [ ]:
# Re-use the quantile-shifted tensors we already built for the MLP.
ftt = FTTransformer(
    n_features=X_tr_s.shape[1],
    d_token=72,
    n_layers=4,
    n_heads=8,
    ffn_factor=2.0,
    dropout=0.04,
).to(DEVICE)

print(f"FT-Transformer parameter count: {sum(p.numel() for p in ftt.parameters()):,}")

optimizer_ftt = torch.optim.AdamW(
    ftt.parameters(),
    lr=1e-4,
    weight_decay=2.38e-6,
)
scheduler_ftt = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ftt, mode="min", factor=0.5, patience=20, min_lr=1e-6,
)
loss_fn = nn.MSELoss()

MAX_EPOCHS_FTT     = 200
EARLY_PATIENCE_FTT = 40

best_val, best_state, epochs_no_imp = float("inf"), None, 0
history_ftt = {"train_loss": [], "val_loss": [], "lr": []}

for epoch in range(1, MAX_EPOCHS_FTT + 1):
    ftt.train()
    epoch_loss, n = 0.0, 0
    for xb, yb in train_loader:          # same DataLoader as MLP
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_ftt.zero_grad()
        pred = ftt(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        # modest gradient clipping — transformers can blow up early
        torch.nn.utils.clip_grad_norm_(ftt.parameters(), max_norm=1.0)
        optimizer_ftt.step()
        epoch_loss += loss.item() * len(xb); n += len(xb)
    train_loss = epoch_loss / n

    ftt.eval()
    with torch.no_grad():
        val_loss = loss_fn(ftt(val_tensor_X), val_tensor_y).item()

    history_ftt["train_loss"].append(train_loss)
    history_ftt["val_loss"].append(val_loss)
    history_ftt["lr"].append(optimizer_ftt.param_groups[0]["lr"])
    scheduler_ftt.step(val_loss)

    if val_loss < best_val - 1e-5:
        best_val, epochs_no_imp = val_loss, 0
        best_state = {k: v.cpu().clone() for k, v in ftt.state_dict().items()}
    else:
        epochs_no_imp += 1

    if epoch == 1 or epoch % 10 == 0 or epoch == MAX_EPOCHS_FTT:
        print(f"epoch {epoch:3d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}"
              f"  lr={optimizer_ftt.param_groups[0]['lr']:.2e}")

    if epochs_no_imp >= EARLY_PATIENCE_FTT:
        print(f"Early stopping at epoch {epoch}")
        break

ftt.load_state_dict(best_state)
print(f"Best val loss (shifted-quantile MSE) = {best_val:.5f}")

Plot FT-Transformer training curves.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_ftt["train_loss"], label="train")
ax[0].plot(history_ftt["val_loss"],   label="val")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("MSE (quantile-shift space)")
ax[0].set_title("FT-Transformer loss"); ax[0].legend()
ax[1].plot(history_ftt["lr"])
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("learning rate")
ax[1].set_title("LR schedule"); ax[1].set_yscale("log")
plt.tight_layout(); plt.show()

Score the FT-Transformer on train & validation, and cache its test-set predictions in quantile space.

In [ ]:
@torch.no_grad()
def ftt_predict_orig(X_scaled):
    ftt.eval()
    t = torch.from_numpy(X_scaled).float().to(DEVICE)
    out = []
    for i in range(0, len(t), 2048):
        out.append(ftt(t[i:i+2048]).cpu().numpy())
    y_shift = np.concatenate(out)
    y_q     = y_shift - Y_SHIFT
    return inverse_target(y_q)

pred_tr_orig  = ftt_predict_orig(X_tr_s)
pred_val_orig = ftt_predict_orig(X_val_s)

results["FT-Transformer"] = {
    "train": evaluate(y_tr,  pred_tr_orig,  "FT-Transformer (train)"),
    "val":   evaluate(y_val, pred_val_orig, "FT-Transformer (val)  "),
}

_t_test = torch.from_numpy(X_test_s).float().to(DEVICE)
with torch.no_grad():
    ftt_test_pred_q_list = []
    for i in range(0, len(_t_test), 2048):
        ftt_test_pred_q_list.append(ftt(_t_test[i:i+2048]).cpu().numpy())
    ftt_test_pred_q = np.concatenate(ftt_test_pred_q_list) - Y_SHIFT

## 9. Model comparison

All three metrics on training and validation sets. MAPE is the primary metric — *lower
is better*. R² closer to 1 is better.

In [ ]:
rows = []
for name, splits in results.items():
    for split_name, m in splits.items():
        rows.append({
            "Model":   name,
            "Split":   split_name,
            "MAPE":    m["MAPE"],
            "R²":      m["R2"],
            "RMSE":    m["RMSE"],
        })
comparison = pd.DataFrame(rows)

# Pivot so each metric has a clear train/val pair per model
pivot_mape = comparison.pivot(index="Model", columns="Split", values="MAPE").add_prefix("MAPE_")
pivot_r2   = comparison.pivot(index="Model", columns="Split", values="R²").add_prefix("R²_")
pivot_rmse = comparison.pivot(index="Model", columns="Split", values="RMSE").add_prefix("RMSE_")
summary = pd.concat([pivot_mape, pivot_r2, pivot_rmse], axis=1)
summary = summary.sort_values("MAPE_val")

pd.options.display.float_format = "{:,.4f}".format
print("Model comparison (sorted by validation MAPE — lower is better):")
summary

Pick the winning model — the one with the lowest validation MAPE.

In [ ]:
# Pick the best model by validation MAPE
best_model_name = summary["MAPE_val"].idxmin()
print(f"Best model by validation MAPE: {best_model_name}")

## 10. Final predictions & export

Produce test-set predictions with the best model, inverse-transform from quantile space
to bar, and write `prediction.csv` matching `sample_prediction.csv` exactly.

In [ ]:
best_test_q = {
    "Ridge":           ridge_test_pred_q,
    "XGBoost":         xgb_test_pred_q,
    "MLP":             mlp_test_pred_q,
    "FT-Transformer":  ftt_test_pred_q,
}[best_model_name]

test_pred_orig = inverse_target(best_test_q)

# Sanity: predictions must be positive. Clip below zero just in case.
test_pred_orig = np.clip(test_pred_orig, a_min=0.0, a_max=None)

submission = pd.DataFrame({
    "ID": test_ids.astype(int),
    "Target Pressure (bar)": test_pred_orig,
})

# Match the sample's column order and dtypes
assert list(submission.columns) == list(sample.columns), "Column mismatch!"
assert len(submission) == len(sample), f"Row count mismatch: {len(submission)} vs {len(sample)}"

OUT_PATH = "prediction.csv"
submission.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}  ({len(submission)} rows)")
print("\nFirst 5 rows:")
print(submission.head().to_string(index=False))
print("\nSample file for reference:")
print(sample.head().to_string(index=False))

In [ ]:
# Per-model submissions + simple ensemble
import numpy as np
import pandas as pd

all_preds_orig = {
    "Ridge":          np.clip(inverse_target(ridge_test_pred_q),  0, None),
    "XGBoost":        np.clip(inverse_target(xgb_test_pred_q),    0, None),
    "MLP":            np.clip(inverse_target(mlp_test_pred_q),    0, None),
    "FT-Transformer": np.clip(inverse_target(ftt_test_pred_q),    0, None),
}

# 1) one CSV per model
for name, preds in all_preds_orig.items():
    fname = f"prediction_{name.lower().replace('-', '_')}.csv"
    pd.DataFrame({
        "ID": test_ids.astype(int),
        "Target Pressure (bar)": preds,
    }).to_csv(fname, index=False)
    print(f"wrote {fname}")

# 2) Ensemble of the two strongest models (based on val MAPE)
top2 = summary["MAPE_val"].nsmallest(2).index.tolist()
print(f"\nEnsembling top 2 by validation MAPE: {top2}")
ensemble_preds = np.sqrt(all_preds_orig[top2[0]] * all_preds_orig[top2[1]])
pd.DataFrame({
    "ID": test_ids.astype(int),
    "Target Pressure (bar)": ensemble_preds,
}).to_csv("prediction_ensemble.csv", index=False)
print("wrote prediction_ensemble.csv")

### Done.

`prediction.csv` is ready to submit. Re-run the whole notebook from top to reproduce
(all random states are seeded to 42).